In [ ]:
import requests
import pandas as pd
import time
import json
import os
import gc
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import glob

# ── Resilient session with connection pooling ──
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20,
                      pool_maxsize=20)
session.mount("https://", adapter)
API_BASE = "https://arctic-shift.photon-reddit.com/api"

from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = [
    "amitheasshole",
]

MIN_COMMENTS = 5
REQUEST_DELAY = 0.1

# Outer chunks run sequentially — inner years run in parallel
# Workers per chunk = number of years in that chunk
YEAR_RANGE_CHUNKS = [
    # (2010, 2014),   # 4 workers: 2010, 2011, 2012, 2013
    # (2014, 2016),   # 2 workers: 2014, 2015
    # (2016, 2018),   # 2 workers: 2016, 2017
    # (2018, 2020),   # 2 workers: 2018, 2019
    (2020, 2022),   # 2 workers: 2020, 2021
    (2022, 2026),   # 4 workers: 2022, 2023, 2024, 2025
]

POST_FIELDS = [
    "id", "author", "subreddit", "title", "selftext", "score",
    "created_utc", "num_comments", "url", "over_18",
    "link_flair_text", "author_flair_text", "edited",
]
COMMENT_FIELDS = [
    "id", "author", "subreddit", "body", "score", "created_utc",
    "link_id", "parent_id", "distinguished", "author_flair_text",
    "edited", "controversiality",
]

print_lock = threading.Lock()
def tprint(msg):
    with print_lock:
        print(msg)

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════

def chunk_path(subreddit, endpoint_type, y_start, y_end):
    """Path for a single chunk CSV: e.g. chunks/changemyview_comments_2020-2022.csv"""
    return os.path.join(CHUNKS_DIR, f"{subreddit}_{endpoint_type}_{y_start}-{y_end}.csv")


def paginate_and_save(endpoint, subreddit, fields, y_start, y_end,
                      endpoint_type, label="items"):
    """
    Paginate one time-range chunk. Saves directly to CSV.
    Returns (chunk_label, row_count) or skips if file exists.
    """
    chunk_label = f"{y_start}-{y_end}"
    out_path = chunk_path(subreddit, endpoint_type, y_start, y_end)

    # ── Resume: skip if chunk already saved ──
    if os.path.exists(out_path):
        try:
            existing = pd.read_csv(out_path, nrows=1)
            row_count = sum(1 for _ in open(out_path)) - 1  # minus header
            tprint(f"      ⏭️  [{chunk_label}] {row_count:,} {label} already on disk, skipping")
            return chunk_label, row_count
        except Exception:
            pass  # Corrupted, re-fetch

    results = []
    params = {
        "subreddit": subreddit,
        "sort": "asc",
        "limit": 100,
        "after": f"{y_start}-01-01",
        "before": f"{y_end}-01-01",
    }
    total = 0
    t0 = time.time()
    stall_count = 0
    last_log = t0

    while True:
        try:
            resp = session.get(f"{API_BASE}/{endpoint}",
                               params=params, timeout=60)

            remaining = resp.headers.get("X-RateLimit-Remaining")
            if remaining is not None and int(remaining) < 200:
                reset = resp.headers.get("X-RateLimit-Reset", "5")
                wait = max(float(reset), 2.0)
                tprint(f"      ⏳ [{chunk_label}] Rate limit "
                       f"({remaining} left), waiting {wait:.0f}s")
                time.sleep(wait)

            if resp.status_code == 429:
                tprint(f"      ⏳ [{chunk_label}] 429, backing off 15s")
                time.sleep(15)
                continue

            resp.raise_for_status()
            data = resp.json().get("data", []) or []

        except Exception as e:
            stall_count += 1
            if stall_count > 10:
                tprint(f"      ❌ [{chunk_label}] Too many errors, "
                       f"stopping at {total:,}")
                break
            time.sleep(5)
            continue

        stall_count = 0

        if not data:
            break

        for item in data:
            results.append({f: item.get(f) for f in fields})
            total += 1

        last_ts = data[-1].get("created_utc")
        if last_ts is None:
            break
        params["after"] = last_ts

        now = time.time()
        if now - last_log > 30:
            elapsed = now - t0
            rate = total / elapsed if elapsed > 0 else 0
            tprint(f"      [{chunk_label}] {total:,} {label} "
                   f"({rate:.0f}/s, {elapsed/60:.1f}m)")
            last_log = now

        if len(data) < 100:
            break

        time.sleep(REQUEST_DELAY)

    # ── Save chunk to CSV ──
    elapsed = time.time() - t0
    if results:
        df = pd.DataFrame(results)
        df.drop_duplicates(subset=["id"], inplace=True)
        df.to_csv(out_path, index=False)
        tprint(f"      ✅ [{chunk_label}] {len(df):,} {label} saved in "
               f"{elapsed/60:.1f} min → {out_path}")
        saved = len(df)
        del df
    else:
        # Save empty CSV with headers so resume knows it's done
        pd.DataFrame(columns=fields).to_csv(out_path, index=False)
        tprint(f"      ✅ [{chunk_label}] 0 {label} in {elapsed/60:.1f} min")
        saved = 0

    del results
    gc.collect()
    return chunk_label, saved


def fetch_all_chunks(endpoint, subreddit, fields, endpoint_type, label="items"):
    """
    For each outer range (e.g. 2010-2014), expand into individual years
    (2010, 2011, 2012, 2013) and fetch them in parallel.
    Outer ranges run sequentially. Inner years run in parallel.
    Each year saves its own CSV.
    """
    all_counts = {}

    for range_start, range_end in YEAR_RANGE_CHUNKS:
        individual_years = [(y, y + 1) for y in range(range_start, range_end)]
        n_workers = len(individual_years)

        # Check if ALL years in this range are already done
        all_done = all(
            os.path.exists(chunk_path(subreddit, endpoint_type, y, y + 1))
            for y, _ in individual_years
        )
        if all_done:
            tprint(f"    ⏭️  [{range_start}-{range_end}] all {n_workers} "
                   f"year(s) already on disk")
            for y, y1 in individual_years:
                p = chunk_path(subreddit, endpoint_type, y, y1)
                try:
                    row_count = sum(1 for _ in open(p)) - 1
                except Exception:
                    row_count = 0
                all_counts[f"{y}-{y1}"] = max(row_count, 0)
            continue

        tprint(f"    📂 [{range_start}-{range_end}] launching {n_workers} "
               f"year(s) in parallel")

        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {}
            for y_start, y_end in individual_years:
                future = executor.submit(
                    paginate_and_save, endpoint, subreddit, fields,
                    y_start, y_end, endpoint_type, label
                )
                futures[future] = f"{y_start}-{y_end}"

            for future in as_completed(futures):
                chunk_label = futures[future]
                try:
                    label_out, count = future.result()
                    all_counts[label_out] = count
                    tprint(f"    📦 [{label_out}] {count:,} {label}")
                except Exception as e:
                    tprint(f"    ❌ [{chunk_label}] failed: {e}")
                    all_counts[chunk_label] = 0

    return all_counts

def merge_chunks(subreddit, endpoint_type, fields):
    """
    Merge all chunk CSVs for a subreddit+type into one DataFrame.
    """
    pattern = os.path.join(CHUNKS_DIR, f"{subreddit}_{endpoint_type}_*.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        return pd.DataFrame(columns=fields)

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, low_memory=False)
            if len(df) > 0:
                dfs.append(df)
        except Exception as e:
            tprint(f"    ⚠️  Couldn't read {f}: {e}")

    if not dfs:
        return pd.DataFrame(columns=fields)

    merged = pd.concat(dfs, ignore_index=True)
    merged.drop_duplicates(subset=["id"], inplace=True)
    del dfs
    gc.collect()
    return merged


def filter_posts(df):
    if len(df) == 0:
        return df
    selftext = df["selftext"].fillna("")
    author = df["author"].fillna("").str.lower()
    num_comments = df["num_comments"].fillna(0)
    mask = (
        ~selftext.isin(["[deleted]", "[removed]", ""]) &
        ~author.isin(["automoderator", "[deleted]", ""]) &
        (num_comments >= MIN_COMMENTS)
    )
    return df[mask].copy()


def filter_comments(df):
    if len(df) == 0:
        return df
    body = df["body"].fillna("")
    author_lower = df["author"].fillna("").str.lower()
    df["is_deltabot"] = author_lower == "deltabot"
    mask = (
        ~body.isin(["[deleted]", "[removed]", ""]) &
        ~author_lower.isin(["automoderator", "[deleted]", ""])
    )
    return df[mask].copy()


# ═══════════════════════════════════════════════════════════════
# MAIN SCRAPE LOOP
# ═══════════════════════════════════════════════════════════════

overall_start = time.time()
scrape_logs = []

for sub in SUBREDDITS:
    sub_start = time.time()

    # ── Resume: skip if final merged files exist ──
    posts_path = os.path.join(OUT_DIR, f"{sub}_posts.csv")
    comments_path = os.path.join(OUT_DIR, f"{sub}_comments.csv")
    if os.path.exists(posts_path) and os.path.exists(comments_path):
        try:
            ep = pd.read_csv(posts_path, nrows=2)
            ec = pd.read_csv(comments_path, nrows=2)
            if len(ep) > 0 and len(ec) > 0:
                print(f"\n⏭️  r/{sub} — already merged, skipping.")
                continue
        except Exception:
            pass

    print(f"\n{'='*65}")
    print(f"  r/{sub}  — parallel chunked API scrape")
    print(f"{'='*65}")

    # ── Fetch posts (parallel, each chunk saves its own CSV) ──
    print(f"\n  📡 Fetching posts ...")
    post_counts = fetch_all_chunks("posts/search", sub, POST_FIELDS,
                                    "posts", "posts")
    total_raw_posts = sum(post_counts.values())
    print(f"    Total raw posts across chunks: {total_raw_posts:,}")

    # ── Fetch comments (parallel, each chunk saves its own CSV) ──
    print(f"\n  📡 Fetching comments ...")
    comment_counts = fetch_all_chunks("comments/search", sub, COMMENT_FIELDS,
                                       "comments", "comments")
    total_raw_comments = sum(comment_counts.values())
    print(f"    Total raw comments across chunks: {total_raw_comments:,}")

    # ── Merge + filter + save ──
    print(f"\n  🔗 Merging and filtering ...")

    posts_df = merge_chunks(sub, "posts", POST_FIELDS)
    print(f"    Merged unique posts: {len(posts_df):,}")
    valid_posts = filter_posts(posts_df)
    print(f"    After filtering: {len(valid_posts):,} / {len(posts_df):,}")
    del posts_df
    gc.collect()

    comments_df = merge_chunks(sub, "comments", COMMENT_FIELDS)
    print(f"    Merged unique comments: {len(comments_df):,}")
    valid_comments = filter_comments(comments_df)
    deltabot_count = (valid_comments["is_deltabot"].sum()
                      if "is_deltabot" in valid_comments.columns else 0)
    print(f"    After filtering: {len(valid_comments):,} / {len(comments_df):,}")
    if deltabot_count > 0:
        print(f"    🏅 DeltaBot flagged: {deltabot_count:,}")
    del comments_df
    gc.collect()

    # ── Link comments to valid posts ──
    valid_posts["id"] = valid_posts["id"].astype(str)
    valid_post_ids = set("t3_" + valid_posts["id"])
    valid_comments["link_id"] = (valid_comments["link_id"]
                                  .fillna("").astype(str))
    valid_comments = valid_comments[
        valid_comments["link_id"].isin(valid_post_ids)
    ].copy()
    print(f"    Comments for valid posts: {len(valid_comments):,}")

    # ── Add post_author ──
    post_author_map = valid_posts.set_index("id")["author"].to_dict()
    valid_comments["post_id"] = (
        valid_comments["link_id"].str.replace("t3_", "", regex=False)
    )
    valid_comments["post_author"] = (
        valid_comments["post_id"].map(post_author_map).fillna("")
    )

    # ── Delta detection ──
    if "is_deltabot" in valid_comments.columns:
        deltabot_rows = valid_comments[valid_comments["is_deltabot"]]
        deltabot_post_ids = set(deltabot_rows["post_id"])
        # Identify which comments earned deltas (parent of DeltaBot reply)
        delta_awarded_comment_ids = set(
            deltabot_rows["parent_id"].str.replace("t1_", "", regex=False)
        )
        valid_comments["delta_awarded"] = valid_comments["id"].isin(
            delta_awarded_comment_ids
        )
        n_before = len(valid_comments)
        valid_comments = valid_comments[
            ~valid_comments["is_deltabot"]
        ].copy()
        valid_comments.drop(columns=["is_deltabot"], inplace=True,
                            errors="ignore")
        dropped = n_before - len(valid_comments)
        if dropped > 0:
            print(f"    Dropped {dropped:,} DeltaBot rows")
        n_delta_comments = valid_comments["delta_awarded"].sum()
        if n_delta_comments > 0:
            print(f"    🏅 Delta-awarded comments: {n_delta_comments:,}")
    else:
        deltabot_post_ids = set()
        valid_comments["delta_awarded"] = False
    valid_posts["has_delta"] = valid_posts["id"].isin(deltabot_post_ids)

    # ── Save final merged files ──
    valid_posts.to_csv(posts_path, index=False)
    valid_comments.to_csv(comments_path, index=False)

    sub_elapsed = (time.time() - sub_start) / 60

    print(f"\n    ✅ r/{sub} — {sub_elapsed:.1f} min")
    print(f"    Posts:    {len(valid_posts):,}")
    print(f"    Comments: {len(valid_comments):,}")

    if len(valid_comments) > 0:
        n_edited = valid_comments["edited"].notna().sum()
        n_controv = valid_comments["controversiality"].notna().sum()
        n_delta = int(valid_posts["has_delta"].sum())
        unique_authors = valid_comments["author"].nunique()
        top_level = (valid_comments["parent_id"].astype(str)
                     .str.startswith("t3_").sum())
        replies = (valid_comments["parent_id"].astype(str)
                   .str.startswith("t1_").sum())

        print(f"    Top-level: {top_level:,} | Replies: {replies:,}")
        print(f"    Unique authors:   {unique_authors:,}")
        print(f"    edited:           {n_edited:,}")
        print(f"    controversiality: {n_controv:,}")
        print(f"    has_delta:        {n_delta:,}")
    else:
        unique_authors = 0
        n_edited = n_controv = n_delta = 0

    log_entry = {
        "subreddit": sub,
        "scrape_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "method": "arctic_shift_api_chunked_with_resume",
        "valid_posts": len(valid_posts),
        "total_comments": len(valid_comments),
        "deltabot_posts": n_delta,
        "unique_comment_authors": int(unique_authors),
        "has_edited": int(n_edited),
        "has_controversiality": int(n_controv),
        "elapsed_minutes": round(sub_elapsed, 1),
    }
    scrape_logs.append(log_entry)
    with open(os.path.join(OUT_DIR, f"{sub}_scrape_log.json"), "w") as f:
        json.dump(log_entry, f, indent=2)

    del valid_posts, valid_comments
    gc.collect()


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════

total_elapsed = (time.time() - overall_start) / 60

print(f"\n\n{'='*65}")
print(f"  COMPLETE — {total_elapsed:.1f} minutes")
print(f"{'='*65}\n")

summary_rows = []
for log in scrape_logs:
    summary_rows.append({
        "Subreddit": f"r/{log['subreddit']}",
        "Posts": f"{log['valid_posts']:,}",
        "Comments": f"{log['total_comments']:,}",
        "Deltas": f"{log['deltabot_posts']:,}",
        "Authors": f"{log['unique_comment_authors']:,}",
        "edited": f"{log['has_edited']:,}",
        "controv.": f"{log['has_controversiality']:,}",
        "Min": log["elapsed_minutes"],
    })

summary_df = pd.DataFrame(summary_rows)
total_c = sum(l["total_comments"] for l in scrape_logs)
total_p = sum(l["valid_posts"] for l in scrape_logs)

print(summary_df.to_string(index=False))
print(f"\nTOTAL: {total_p:,} posts, {total_c:,} comments")

if total_c < 500_000:
    print(f"\n⚠️  {total_c:,} comments — below 3M target.")
elif total_c < 2_000_000:
    print(f"\n📊 {total_c:,} comments — decent, below 3M.")
else:
    print(f"\n✅ {total_c:,} comments — on track for 3M!")

with open(os.path.join(OUT_DIR, "scrape_log_all.json"), "w") as f:
    json.dump(scrape_logs, f, indent=2)

print(f"\n✅ All fields native — no backfill needed")
print(f"Files saved to: {OUT_DIR}")
print(f"Chunk files in: {CHUNKS_DIR}")

Mounted at /content/drive

  r/amitheasshole  — parallel chunked API scrape

  📡 Fetching posts ...
    ⏭️  [2020-2022] all 2 year(s) already on disk
    ⏭️  [2022-2026] all 4 year(s) already on disk
    Total raw posts across chunks: 3,681

  📡 Fetching comments ...
    📂 [2020-2022] launching 2 year(s) in parallel
      [2021-2022] 5,200 comments (172/s, 0.5m)
      [2020-2021] 4,900 comments (161/s, 0.5m)
      [2021-2022] 10,300 comments (170/s, 1.0m)
      [2020-2021] 9,900 comments (162/s, 1.0m)
      [2021-2022] 15,400 comments (169/s, 1.5m)
      [2020-2021] 14,900 comments (163/s, 1.5m)
      [2020-2021] 20,000 comments (165/s, 2.0m)
      [2021-2022] 20,500 comments (168/s, 2.0m)
      [2020-2021] 25,300 comments (167/s, 2.5m)
      [2021-2022] 25,600 comments (168/s, 2.5m)
      [2020-2021] 30,300 comments (166/s, 3.0m)
      [2021-2022] 30,800 comments (169/s, 3.0m)
      [2020-2021] 35,300 comments (166/s, 3.5m)
      [2021-2022] 35,700 comments (168/s, 3.5m)
      [2021-2

In [ ]:
import requests
import pandas as pd
import time
import json
import os
import gc
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import glob

# ── Resilient session with connection pooling ──
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20,
                      pool_maxsize=20)
session.mount("https://", adapter)
API_BASE = "https://arctic-shift.photon-reddit.com/api"

from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = [
    "amitheasshole",
]

MIN_COMMENTS = 5
REQUEST_DELAY = 0.1

# Outer chunks run sequentially — inner years run in parallel
# Workers per chunk = number of years in that chunk
YEAR_RANGE_CHUNKS = [
    # (2010, 2014),   # 4 workers: 2010, 2011, 2012, 2013
    # (2014, 2016),   # 2 workers: 2014, 2015
    # (2016, 2018),   # 2 workers: 2016, 2017
    # (2018, 2020),   # 2 workers: 2018, 2019
    # (2020, 2022),   # 2 workers: 2020, 2021
    (2022, 2026),   # 4 workers: 2022, 2023, 2024, 2025
]

POST_FIELDS = [
    "id", "author", "subreddit", "title", "selftext", "score",
    "created_utc", "num_comments", "url", "over_18",
    "link_flair_text", "author_flair_text", "edited",
]
COMMENT_FIELDS = [
    "id", "author", "subreddit", "body", "score", "created_utc",
    "link_id", "parent_id", "distinguished", "author_flair_text",
    "edited", "controversiality",
]

print_lock = threading.Lock()
def tprint(msg):
    with print_lock:
        print(msg)

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════

def chunk_path(subreddit, endpoint_type, y_start, y_end):
    """Path for a single chunk CSV: e.g. chunks/changemyview_comments_2020-2022.csv"""
    return os.path.join(CHUNKS_DIR, f"{subreddit}_{endpoint_type}_{y_start}-{y_end}.csv")


def paginate_and_save(endpoint, subreddit, fields, y_start, y_end,
                      endpoint_type, label="items"):
    """
    Paginate one time-range chunk. Saves directly to CSV.
    Returns (chunk_label, row_count) or skips if file exists.
    """
    chunk_label = f"{y_start}-{y_end}"
    out_path = chunk_path(subreddit, endpoint_type, y_start, y_end)

    # ── Resume: skip if chunk already saved ──
    if os.path.exists(out_path):
        try:
            existing = pd.read_csv(out_path, nrows=1)
            row_count = sum(1 for _ in open(out_path)) - 1  # minus header
            tprint(f"      ⏭️  [{chunk_label}] {row_count:,} {label} already on disk, skipping")
            return chunk_label, row_count
        except Exception:
            pass  # Corrupted, re-fetch

    results = []
    params = {
        "subreddit": subreddit,
        "sort": "asc",
        "limit": 100,
        "after": f"{y_start}-01-01",
        "before": f"{y_end}-01-01",
    }
    total = 0
    t0 = time.time()
    stall_count = 0
    last_log = t0

    while True:
        try:
            resp = session.get(f"{API_BASE}/{endpoint}",
                               params=params, timeout=60)

            remaining = resp.headers.get("X-RateLimit-Remaining")
            if remaining is not None and int(remaining) < 200:
                reset = resp.headers.get("X-RateLimit-Reset", "5")
                wait = max(float(reset), 2.0)
                tprint(f"      ⏳ [{chunk_label}] Rate limit "
                       f"({remaining} left), waiting {wait:.0f}s")
                time.sleep(wait)

            if resp.status_code == 429:
                tprint(f"      ⏳ [{chunk_label}] 429, backing off 15s")
                time.sleep(15)
                continue

            resp.raise_for_status()
            data = resp.json().get("data", []) or []

        except Exception as e:
            stall_count += 1
            if stall_count > 10:
                tprint(f"      ❌ [{chunk_label}] Too many errors, "
                       f"stopping at {total:,}")
                break
            time.sleep(5)
            continue

        stall_count = 0

        if not data:
            break

        for item in data:
            results.append({f: item.get(f) for f in fields})
            total += 1

        last_ts = data[-1].get("created_utc")
        if last_ts is None:
            break
        params["after"] = last_ts

        now = time.time()
        if now - last_log > 30:
            elapsed = now - t0
            rate = total / elapsed if elapsed > 0 else 0
            tprint(f"      [{chunk_label}] {total:,} {label} "
                   f"({rate:.0f}/s, {elapsed/60:.1f}m)")
            last_log = now

        if len(data) < 100:
            break

        time.sleep(REQUEST_DELAY)

    # ── Save chunk to CSV ──
    elapsed = time.time() - t0
    if results:
        df = pd.DataFrame(results)
        df.drop_duplicates(subset=["id"], inplace=True)
        df.to_csv(out_path, index=False)
        tprint(f"      ✅ [{chunk_label}] {len(df):,} {label} saved in "
               f"{elapsed/60:.1f} min → {out_path}")
        saved = len(df)
        del df
    else:
        # Save empty CSV with headers so resume knows it's done
        pd.DataFrame(columns=fields).to_csv(out_path, index=False)
        tprint(f"      ✅ [{chunk_label}] 0 {label} in {elapsed/60:.1f} min")
        saved = 0

    del results
    gc.collect()
    return chunk_label, saved


def fetch_all_chunks(endpoint, subreddit, fields, endpoint_type, label="items"):
    """
    For each outer range (e.g. 2010-2014), expand into individual years
    (2010, 2011, 2012, 2013) and fetch them in parallel.
    Outer ranges run sequentially. Inner years run in parallel.
    Each year saves its own CSV.
    """
    all_counts = {}

    for range_start, range_end in YEAR_RANGE_CHUNKS:
        individual_years = [(y, y + 1) for y in range(range_start, range_end)]
        n_workers = len(individual_years)

        # Check if ALL years in this range are already done
        all_done = all(
            os.path.exists(chunk_path(subreddit, endpoint_type, y, y + 1))
            for y, _ in individual_years
        )
        if all_done:
            tprint(f"    ⏭️  [{range_start}-{range_end}] all {n_workers} "
                   f"year(s) already on disk")
            for y, y1 in individual_years:
                p = chunk_path(subreddit, endpoint_type, y, y1)
                try:
                    row_count = sum(1 for _ in open(p)) - 1
                except Exception:
                    row_count = 0
                all_counts[f"{y}-{y1}"] = max(row_count, 0)
            continue

        tprint(f"    📂 [{range_start}-{range_end}] launching {n_workers} "
               f"year(s) in parallel")

        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {}
            for y_start, y_end in individual_years:
                future = executor.submit(
                    paginate_and_save, endpoint, subreddit, fields,
                    y_start, y_end, endpoint_type, label
                )
                futures[future] = f"{y_start}-{y_end}"

            for future in as_completed(futures):
                chunk_label = futures[future]
                try:
                    label_out, count = future.result()
                    all_counts[label_out] = count
                    tprint(f"    📦 [{label_out}] {count:,} {label}")
                except Exception as e:
                    tprint(f"    ❌ [{chunk_label}] failed: {e}")
                    all_counts[chunk_label] = 0

    return all_counts

def merge_chunks(subreddit, endpoint_type, fields):
    """
    Merge all chunk CSVs for a subreddit+type into one DataFrame.
    """
    pattern = os.path.join(CHUNKS_DIR, f"{subreddit}_{endpoint_type}_*.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        return pd.DataFrame(columns=fields)

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, low_memory=False)
            if len(df) > 0:
                dfs.append(df)
        except Exception as e:
            tprint(f"    ⚠️  Couldn't read {f}: {e}")

    if not dfs:
        return pd.DataFrame(columns=fields)

    merged = pd.concat(dfs, ignore_index=True)
    merged.drop_duplicates(subset=["id"], inplace=True)
    del dfs
    gc.collect()
    return merged


def filter_posts(df):
    if len(df) == 0:
        return df
    selftext = df["selftext"].fillna("")
    author = df["author"].fillna("").str.lower()
    num_comments = df["num_comments"].fillna(0)
    mask = (
        ~selftext.isin(["[deleted]", "[removed]", ""]) &
        ~author.isin(["automoderator", "[deleted]", ""]) &
        (num_comments >= MIN_COMMENTS)
    )
    return df[mask].copy()


def filter_comments(df):
    if len(df) == 0:
        return df
    body = df["body"].fillna("")
    author_lower = df["author"].fillna("").str.lower()
    df["is_deltabot"] = author_lower == "deltabot"
    mask = (
        ~body.isin(["[deleted]", "[removed]", ""]) &
        ~author_lower.isin(["automoderator", "[deleted]", ""])
    )
    return df[mask].copy()


# ═══════════════════════════════════════════════════════════════
# MAIN SCRAPE LOOP
# ═══════════════════════════════════════════════════════════════

overall_start = time.time()
scrape_logs = []

for sub in SUBREDDITS:
    sub_start = time.time()

    # ── Resume: skip if final merged files exist ──
    posts_path = os.path.join(OUT_DIR, f"{sub}_posts.csv")
    comments_path = os.path.join(OUT_DIR, f"{sub}_comments.csv")
    if os.path.exists(posts_path) and os.path.exists(comments_path):
        try:
            ep = pd.read_csv(posts_path, nrows=2)
            ec = pd.read_csv(comments_path, nrows=2)
            if len(ep) > 0 and len(ec) > 0:
                print(f"\n⏭️  r/{sub} — already merged, skipping.")
                continue
        except Exception:
            pass

    print(f"\n{'='*65}")
    print(f"  r/{sub}  — parallel chunked API scrape")
    print(f"{'='*65}")

    # ── Fetch posts (parallel, each chunk saves its own CSV) ──
    print(f"\n  📡 Fetching posts ...")
    post_counts = fetch_all_chunks("posts/search", sub, POST_FIELDS,
                                    "posts", "posts")
    total_raw_posts = sum(post_counts.values())
    print(f"    Total raw posts across chunks: {total_raw_posts:,}")

    # ── Fetch comments (parallel, each chunk saves its own CSV) ──
    print(f"\n  📡 Fetching comments ...")
    comment_counts = fetch_all_chunks("comments/search", sub, COMMENT_FIELDS,
                                       "comments", "comments")
    total_raw_comments = sum(comment_counts.values())
    print(f"    Total raw comments across chunks: {total_raw_comments:,}")

    # ── Merge + filter + save ──
    print(f"\n  🔗 Merging and filtering ...")

    posts_df = merge_chunks(sub, "posts", POST_FIELDS)
    print(f"    Merged unique posts: {len(posts_df):,}")
    valid_posts = filter_posts(posts_df)
    print(f"    After filtering: {len(valid_posts):,} / {len(posts_df):,}")
    del posts_df
    gc.collect()

    comments_df = merge_chunks(sub, "comments", COMMENT_FIELDS)
    print(f"    Merged unique comments: {len(comments_df):,}")
    valid_comments = filter_comments(comments_df)
    deltabot_count = (valid_comments["is_deltabot"].sum()
                      if "is_deltabot" in valid_comments.columns else 0)
    print(f"    After filtering: {len(valid_comments):,} / {len(comments_df):,}")
    if deltabot_count > 0:
        print(f"    🏅 DeltaBot flagged: {deltabot_count:,}")
    del comments_df
    gc.collect()

    # ── Link comments to valid posts ──
    valid_posts["id"] = valid_posts["id"].astype(str)
    valid_post_ids = set("t3_" + valid_posts["id"])
    valid_comments["link_id"] = (valid_comments["link_id"]
                                  .fillna("").astype(str))
    valid_comments = valid_comments[
        valid_comments["link_id"].isin(valid_post_ids)
    ].copy()
    print(f"    Comments for valid posts: {len(valid_comments):,}")

    # ── Add post_author ──
    post_author_map = valid_posts.set_index("id")["author"].to_dict()
    valid_comments["post_id"] = (
        valid_comments["link_id"].str.replace("t3_", "", regex=False)
    )
    valid_comments["post_author"] = (
        valid_comments["post_id"].map(post_author_map).fillna("")
    )

    # ── Delta detection ──
    if "is_deltabot" in valid_comments.columns:
        deltabot_rows = valid_comments[valid_comments["is_deltabot"]]
        deltabot_post_ids = set(deltabot_rows["post_id"])
        # Identify which comments earned deltas (parent of DeltaBot reply)
        delta_awarded_comment_ids = set(
            deltabot_rows["parent_id"].str.replace("t1_", "", regex=False)
        )
        valid_comments["delta_awarded"] = valid_comments["id"].isin(
            delta_awarded_comment_ids
        )
        n_before = len(valid_comments)
        valid_comments = valid_comments[
            ~valid_comments["is_deltabot"]
        ].copy()
        valid_comments.drop(columns=["is_deltabot"], inplace=True,
                            errors="ignore")
        dropped = n_before - len(valid_comments)
        if dropped > 0:
            print(f"    Dropped {dropped:,} DeltaBot rows")
        n_delta_comments = valid_comments["delta_awarded"].sum()
        if n_delta_comments > 0:
            print(f"    🏅 Delta-awarded comments: {n_delta_comments:,}")
    else:
        deltabot_post_ids = set()
        valid_comments["delta_awarded"] = False
    valid_posts["has_delta"] = valid_posts["id"].isin(deltabot_post_ids)

    # ── Save final merged files ──
    valid_posts.to_csv(posts_path, index=False)
    valid_comments.to_csv(comments_path, index=False)

    sub_elapsed = (time.time() - sub_start) / 60

    print(f"\n    ✅ r/{sub} — {sub_elapsed:.1f} min")
    print(f"    Posts:    {len(valid_posts):,}")
    print(f"    Comments: {len(valid_comments):,}")

    if len(valid_comments) > 0:
        n_edited = valid_comments["edited"].notna().sum()
        n_controv = valid_comments["controversiality"].notna().sum()
        n_delta = int(valid_posts["has_delta"].sum())
        unique_authors = valid_comments["author"].nunique()
        top_level = (valid_comments["parent_id"].astype(str)
                     .str.startswith("t3_").sum())
        replies = (valid_comments["parent_id"].astype(str)
                   .str.startswith("t1_").sum())

        print(f"    Top-level: {top_level:,} | Replies: {replies:,}")
        print(f"    Unique authors:   {unique_authors:,}")
        print(f"    edited:           {n_edited:,}")
        print(f"    controversiality: {n_controv:,}")
        print(f"    has_delta:        {n_delta:,}")
    else:
        unique_authors = 0
        n_edited = n_controv = n_delta = 0

    log_entry = {
        "subreddit": sub,
        "scrape_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "method": "arctic_shift_api_chunked_with_resume",
        "valid_posts": len(valid_posts),
        "total_comments": len(valid_comments),
        "deltabot_posts": n_delta,
        "unique_comment_authors": int(unique_authors),
        "has_edited": int(n_edited),
        "has_controversiality": int(n_controv),
        "elapsed_minutes": round(sub_elapsed, 1),
    }
    scrape_logs.append(log_entry)
    with open(os.path.join(OUT_DIR, f"{sub}_scrape_log.json"), "w") as f:
        json.dump(log_entry, f, indent=2)

    del valid_posts, valid_comments
    gc.collect()


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════

total_elapsed = (time.time() - overall_start) / 60

print(f"\n\n{'='*65}")
print(f"  COMPLETE — {total_elapsed:.1f} minutes")
print(f"{'='*65}\n")

summary_rows = []
for log in scrape_logs:
    summary_rows.append({
        "Subreddit": f"r/{log['subreddit']}",
        "Posts": f"{log['valid_posts']:,}",
        "Comments": f"{log['total_comments']:,}",
        "Deltas": f"{log['deltabot_posts']:,}",
        "Authors": f"{log['unique_comment_authors']:,}",
        "edited": f"{log['has_edited']:,}",
        "controv.": f"{log['has_controversiality']:,}",
        "Min": log["elapsed_minutes"],
    })

summary_df = pd.DataFrame(summary_rows)
total_c = sum(l["total_comments"] for l in scrape_logs)
total_p = sum(l["valid_posts"] for l in scrape_logs)

print(summary_df.to_string(index=False))
print(f"\nTOTAL: {total_p:,} posts, {total_c:,} comments")

if total_c < 500_000:
    print(f"\n⚠️  {total_c:,} comments — below 3M target.")
elif total_c < 2_000_000:
    print(f"\n📊 {total_c:,} comments — decent, below 3M.")
else:
    print(f"\n✅ {total_c:,} comments — on track for 3M!")

with open(os.path.join(OUT_DIR, "scrape_log_all.json"), "w") as f:
    json.dump(scrape_logs, f, indent=2)

print(f"\n✅ All fields native — no backfill needed")
print(f"Files saved to: {OUT_DIR}")
print(f"Chunk files in: {CHUNKS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  r/amitheasshole  — parallel chunked API scrape

  📡 Fetching posts ...
    ⏭️  [2022-2026] all 4 year(s) already on disk
    Total raw posts across chunks: 0

  📡 Fetching comments ...
    📂 [2022-2026] launching 4 year(s) in parallel
      [2022-2023] 4,700 comments (157/s, 0.5m)
      [2025-2026] 4,200 comments (139/s, 0.5m)
      [2023-2024] 4,900 comments (162/s, 0.5m)
      [2024-2025] 3,900 comments (128/s, 0.5m)
      [2022-2023] 9,600 comments (159/s, 1.0m)
      [2025-2026] 8,700 comments (144/s, 1.0m)
      [2024-2025] 8,700 comments (144/s, 1.0m)
      [2023-2024] 9,900 comments (163/s, 1.0m)
      [2025-2026] 13,200 comments (146/s, 1.5m)
      [2022-2023] 14,600 comments (161/s, 1.5m)
      [2024-2025] 13,400 comments (147/s, 1.5m)
      [2023-2024] 15,100 comments (165/s, 1.5m)
      [2022-2023] 19,600 comments (162/s, 2.0m)
      [2025-2026]

In [ ]:
import requests
import pandas as pd
import time
import json
import os
import gc
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import glob

# ── Resilient session with connection pooling ──
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20,
                      pool_maxsize=20)
session.mount("https://", adapter)
API_BASE = "https://arctic-shift.photon-reddit.com/api"

from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = [
    "amitheasshole",
]

MIN_COMMENTS = 5
REQUEST_DELAY = 0.1

# Outer chunks run sequentially — inner years run in parallel
# Workers per chunk = number of years in that chunk
YEAR_RANGE_CHUNKS = [
    # (2010, 2014),   # 4 workers: 2010, 2011, 2012, 2013
    # (2014, 2016),   # 2 workers: 2014, 2015
    # (2016, 2018),   # 2 workers: 2016, 2017
    # (2018, 2020),   # 2 workers: 2018, 2019
    # (2020, 2022),   # 2 workers: 2020, 2021
    (2022, 2026),   # 4 workers: 2022, 2023, 2024, 2025
]

POST_FIELDS = [
    "id", "author", "subreddit", "title", "selftext", "score",
    "created_utc", "num_comments", "url", "over_18",
    "link_flair_text", "author_flair_text", "edited",
]
COMMENT_FIELDS = [
    "id", "author", "subreddit", "body", "score", "created_utc",
    "link_id", "parent_id", "distinguished", "author_flair_text",
    "edited", "controversiality",
]

print_lock = threading.Lock()
def tprint(msg):
    with print_lock:
        print(msg)

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════

def chunk_path(subreddit, endpoint_type, y_start, y_end):
    """Path for a single chunk CSV: e.g. chunks/changemyview_comments_2020-2022.csv"""
    return os.path.join(CHUNKS_DIR, f"{subreddit}_{endpoint_type}_{y_start}-{y_end}.csv")


def paginate_and_save(endpoint, subreddit, fields, y_start, y_end,
                      endpoint_type, label="items"):

    chunk_label = f"{y_start}-{y_end}"
    out_path = chunk_path(subreddit, endpoint_type, y_start, y_end)
    progress_path = out_path.replace(".csv", "_progress.txt")

    params = {
        "subreddit": subreddit,
        "sort": "asc",
        "limit": 100,
        "after": f"{y_start}-01-01",
        "before": f"{y_end}-01-01",
    }

    total = 0
    t0 = time.time()
    stall_count = 0
    last_log = t0

    BATCH_SIZE = 5000
    results = []

    # ─────────────────────────────────────────────
    # 🔄 RESUME LOGIC
    # ─────────────────────────────────────────────
    if os.path.exists(out_path):
        try:
            # Fast resume via progress file
            if os.path.exists(progress_path):
                with open(progress_path, "r") as f:
                    last_ts = int(f.read().strip())
                    params["after"] = last_ts + 1
                    tprint(f"      🔄 [{chunk_label}] Resuming from ts={last_ts}")
            else:
                # Fallback: scan CSV (slower)
                df_existing = pd.read_csv(out_path, usecols=["created_utc"])
                last_ts = int(df_existing["created_utc"].max())
                params["after"] = last_ts + 1
                tprint(f"      🔄 [{chunk_label}] Resuming from CSV ts={last_ts}")

            # Count existing rows
            total = sum(1 for _ in open(out_path)) - 1

        except Exception as e:
            tprint(f"      ⚠️ [{chunk_label}] Resume failed, restarting: {e}")

    # ─────────────────────────────────────────────
    # 🚀 MAIN LOOP
    # ─────────────────────────────────────────────
    while True:
        try:
            resp = session.get(f"{API_BASE}/{endpoint}",
                               params=params, timeout=60)

            remaining = resp.headers.get("X-RateLimit-Remaining")
            if remaining is not None and int(remaining) < 200:
                reset = resp.headers.get("X-RateLimit-Reset", "5")
                wait = max(float(reset), 2.0)
                tprint(f"      ⏳ [{chunk_label}] Rate limit "
                       f"({remaining} left), waiting {wait:.0f}s")
                time.sleep(wait)

            if resp.status_code == 429:
                tprint(f"      ⏳ [{chunk_label}] 429, backing off 15s")
                time.sleep(15)
                continue

            resp.raise_for_status()
            data = resp.json().get("data", []) or []

        except Exception as e:
            stall_count += 1
            if stall_count > 10:
                tprint(f"      ❌ [{chunk_label}] Too many errors, stopping at {total:,}")
                break
            time.sleep(5)
            continue

        stall_count = 0

        if not data:
            break

        for item in data:
            results.append({f: item.get(f) for f in fields})
            total += 1

        # ─────────────────────────────────────────
        # 💾 SAVE INCREMENTS
        # ─────────────────────────────────────────
        if len(results) >= BATCH_SIZE:
            df = pd.DataFrame(results)
            df.drop_duplicates(subset=["id"], inplace=True)

            df.to_csv(
                out_path,
                mode="a",
                header=not os.path.exists(out_path),
                index=False
            )

            # Save progress
            last_ts = df["created_utc"].max()
            with open(progress_path, "w") as f:
                f.write(str(int(last_ts)))

            tprint(f"      💾 [{chunk_label}] Saved {len(df):,} rows (total {total:,})")

            results.clear()
            del df
            gc.collect()

        last_ts = data[-1].get("created_utc")
        if last_ts is None:
            break

        params["after"] = last_ts + 1  # 🔑 critical fix

        now = time.time()
        if now - last_log > 30:
            elapsed = now - t0
            rate = total / elapsed if elapsed > 0 else 0
            tprint(f"      [{chunk_label}] {total:,} {label} "
                   f"({rate:.0f}/s, {elapsed/60:.1f}m)")
            last_log = now

        if len(data) < 100:
            break

        time.sleep(REQUEST_DELAY)

    # ─────────────────────────────────────────────
    # 🧹 FINAL FLUSH
    # ─────────────────────────────────────────────
    if results:
        df = pd.DataFrame(results)
        df.drop_duplicates(subset=["id"], inplace=True)

        df.to_csv(
            out_path,
            mode="a",
            header=not os.path.exists(out_path),
            index=False
        )

        last_ts = df["created_utc"].max()
        with open(progress_path, "w") as f:
            f.write(str(int(last_ts)))

        tprint(f"      ✅ [{chunk_label}] Final save {len(df):,} rows")

        del df

    elapsed = time.time() - t0
    tprint(f"      🏁 [{chunk_label}] DONE — {total:,} {label} in {elapsed/60:.1f} min")

    gc.collect()
    return chunk_label, total

def fetch_all_chunks(endpoint, subreddit, fields, endpoint_type, label="items"):
    """
    For each outer range (e.g. 2010-2014), expand into individual years
    (2010, 2011, 2012, 2013) and fetch them in parallel.
    Outer ranges run sequentially. Inner years run in parallel.
    Each year saves its own CSV.
    """
    all_counts = {}

    for range_start, range_end in YEAR_RANGE_CHUNKS:
        individual_years = [(y, y + 1) for y in range(range_start, range_end)]
        n_workers = len(individual_years)

        # Check if ALL years in this range are already done
        all_done = all(
            os.path.exists(chunk_path(subreddit, endpoint_type, y, y + 1))
            for y, _ in individual_years
        )
        if all_done:
            tprint(f"    ⏭️  [{range_start}-{range_end}] all {n_workers} "
                   f"year(s) already on disk")
            for y, y1 in individual_years:
                p = chunk_path(subreddit, endpoint_type, y, y1)
                try:
                    row_count = sum(1 for _ in open(p)) - 1
                except Exception:
                    row_count = 0
                all_counts[f"{y}-{y1}"] = max(row_count, 0)
            continue

        tprint(f"    📂 [{range_start}-{range_end}] launching {n_workers} "
               f"year(s) in parallel")

        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {}
            for y_start, y_end in individual_years:
                future = executor.submit(
                    paginate_and_save, endpoint, subreddit, fields,
                    y_start, y_end, endpoint_type, label
                )
                futures[future] = f"{y_start}-{y_end}"

            for future in as_completed(futures):
                chunk_label = futures[future]
                try:
                    label_out, count = future.result()
                    all_counts[label_out] = count
                    tprint(f"    📦 [{label_out}] {count:,} {label}")
                except Exception as e:
                    tprint(f"    ❌ [{chunk_label}] failed: {e}")
                    all_counts[chunk_label] = 0

    return all_counts

def merge_chunks(subreddit, endpoint_type, fields):
    """
    Merge all chunk CSVs for a subreddit+type into one DataFrame.
    """
    pattern = os.path.join(CHUNKS_DIR, f"{subreddit}_{endpoint_type}_*.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        return pd.DataFrame(columns=fields)

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, low_memory=False)
            if len(df) > 0:
                dfs.append(df)
        except Exception as e:
            tprint(f"    ⚠️  Couldn't read {f}: {e}")

    if not dfs:
        return pd.DataFrame(columns=fields)

    merged = pd.concat(dfs, ignore_index=True)
    merged.drop_duplicates(subset=["id"], inplace=True)
    del dfs
    gc.collect()
    return merged


def filter_posts(df):
    if len(df) == 0:
        return df
    selftext = df["selftext"].fillna("")
    author = df["author"].fillna("").str.lower()
    num_comments = df["num_comments"].fillna(0)
    mask = (
        ~selftext.isin(["[deleted]", "[removed]", ""]) &
        ~author.isin(["automoderator", "[deleted]", ""]) &
        (num_comments >= MIN_COMMENTS)
    )
    return df[mask].copy()


def filter_comments(df):
    if len(df) == 0:
        return df
    body = df["body"].fillna("")
    author_lower = df["author"].fillna("").str.lower()
    df["is_deltabot"] = author_lower == "deltabot"
    mask = (
        ~body.isin(["[deleted]", "[removed]", ""]) &
        ~author_lower.isin(["automoderator", "[deleted]", ""])
    )
    return df[mask].copy()


# ═══════════════════════════════════════════════════════════════
# MAIN SCRAPE LOOP
# ═══════════════════════════════════════════════════════════════

overall_start = time.time()
scrape_logs = []

for sub in SUBREDDITS:
    sub_start = time.time()

    # ── Resume: skip if final merged files exist ──
    posts_path = os.path.join(OUT_DIR, f"{sub}_posts.csv")
    comments_path = os.path.join(OUT_DIR, f"{sub}_comments.csv")
    if os.path.exists(posts_path) and os.path.exists(comments_path):
        try:
            ep = pd.read_csv(posts_path, nrows=2)
            ec = pd.read_csv(comments_path, nrows=2)
            if len(ep) > 0 and len(ec) > 0:
                print(f"\n⏭️  r/{sub} — already merged, skipping.")
                continue
        except Exception:
            pass

    print(f"\n{'='*65}")
    print(f"  r/{sub}  — parallel chunked API scrape")
    print(f"{'='*65}")

    # ── Fetch posts (parallel, each chunk saves its own CSV) ──
    print(f"\n  📡 Fetching posts ...")
    post_counts = fetch_all_chunks("posts/search", sub, POST_FIELDS,
                                    "posts", "posts")
    total_raw_posts = sum(post_counts.values())
    print(f"    Total raw posts across chunks: {total_raw_posts:,}")

    # ── Fetch comments (parallel, each chunk saves its own CSV) ──
    print(f"\n  📡 Fetching comments ...")
    comment_counts = fetch_all_chunks("comments/search", sub, COMMENT_FIELDS,
                                       "comments", "comments")
    total_raw_comments = sum(comment_counts.values())
    print(f"    Total raw comments across chunks: {total_raw_comments:,}")

    # ── Merge + filter + save ──
    print(f"\n  🔗 Merging and filtering ...")

    posts_df = merge_chunks(sub, "posts", POST_FIELDS)
    print(f"    Merged unique posts: {len(posts_df):,}")
    valid_posts = filter_posts(posts_df)
    print(f"    After filtering: {len(valid_posts):,} / {len(posts_df):,}")
    del posts_df
    gc.collect()

    comments_df = merge_chunks(sub, "comments", COMMENT_FIELDS)
    print(f"    Merged unique comments: {len(comments_df):,}")
    valid_comments = filter_comments(comments_df)
    deltabot_count = (valid_comments["is_deltabot"].sum()
                      if "is_deltabot" in valid_comments.columns else 0)
    print(f"    After filtering: {len(valid_comments):,} / {len(comments_df):,}")
    if deltabot_count > 0:
        print(f"    🏅 DeltaBot flagged: {deltabot_count:,}")
    del comments_df
    gc.collect()

    # ── Link comments to valid posts ──
    valid_posts["id"] = valid_posts["id"].astype(str)
    valid_post_ids = set("t3_" + valid_posts["id"])
    valid_comments["link_id"] = (valid_comments["link_id"]
                                  .fillna("").astype(str))
    valid_comments = valid_comments[
        valid_comments["link_id"].isin(valid_post_ids)
    ].copy()
    print(f"    Comments for valid posts: {len(valid_comments):,}")

    # ── Add post_author ──
    post_author_map = valid_posts.set_index("id")["author"].to_dict()
    valid_comments["post_id"] = (
        valid_comments["link_id"].str.replace("t3_", "", regex=False)
    )
    valid_comments["post_author"] = (
        valid_comments["post_id"].map(post_author_map).fillna("")
    )

    # ── Delta detection ──
    if "is_deltabot" in valid_comments.columns:
        deltabot_rows = valid_comments[valid_comments["is_deltabot"]]
        deltabot_post_ids = set(deltabot_rows["post_id"])
        # Identify which comments earned deltas (parent of DeltaBot reply)
        delta_awarded_comment_ids = set(
            deltabot_rows["parent_id"].str.replace("t1_", "", regex=False)
        )
        valid_comments["delta_awarded"] = valid_comments["id"].isin(
            delta_awarded_comment_ids
        )
        n_before = len(valid_comments)
        valid_comments = valid_comments[
            ~valid_comments["is_deltabot"]
        ].copy()
        valid_comments.drop(columns=["is_deltabot"], inplace=True,
                            errors="ignore")
        dropped = n_before - len(valid_comments)
        if dropped > 0:
            print(f"    Dropped {dropped:,} DeltaBot rows")
        n_delta_comments = valid_comments["delta_awarded"].sum()
        if n_delta_comments > 0:
            print(f"    🏅 Delta-awarded comments: {n_delta_comments:,}")
    else:
        deltabot_post_ids = set()
        valid_comments["delta_awarded"] = False
    valid_posts["has_delta"] = valid_posts["id"].isin(deltabot_post_ids)

    # ── Save final merged files ──
    valid_posts.to_csv(posts_path, index=False)
    valid_comments.to_csv(comments_path, index=False)

    sub_elapsed = (time.time() - sub_start) / 60

    print(f"\n    ✅ r/{sub} — {sub_elapsed:.1f} min")
    print(f"    Posts:    {len(valid_posts):,}")
    print(f"    Comments: {len(valid_comments):,}")

    if len(valid_comments) > 0:
        n_edited = valid_comments["edited"].notna().sum()
        n_controv = valid_comments["controversiality"].notna().sum()
        n_delta = int(valid_posts["has_delta"].sum())
        unique_authors = valid_comments["author"].nunique()
        top_level = (valid_comments["parent_id"].astype(str)
                     .str.startswith("t3_").sum())
        replies = (valid_comments["parent_id"].astype(str)
                   .str.startswith("t1_").sum())

        print(f"    Top-level: {top_level:,} | Replies: {replies:,}")
        print(f"    Unique authors:   {unique_authors:,}")
        print(f"    edited:           {n_edited:,}")
        print(f"    controversiality: {n_controv:,}")
        print(f"    has_delta:        {n_delta:,}")
    else:
        unique_authors = 0
        n_edited = n_controv = n_delta = 0

    log_entry = {
        "subreddit": sub,
        "scrape_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "method": "arctic_shift_api_chunked_with_resume",
        "valid_posts": len(valid_posts),
        "total_comments": len(valid_comments),
        "deltabot_posts": n_delta,
        "unique_comment_authors": int(unique_authors),
        "has_edited": int(n_edited),
        "has_controversiality": int(n_controv),
        "elapsed_minutes": round(sub_elapsed, 1),
    }
    scrape_logs.append(log_entry)
    with open(os.path.join(OUT_DIR, f"{sub}_scrape_log.json"), "w") as f:
        json.dump(log_entry, f, indent=2)

    del valid_posts, valid_comments
    gc.collect()


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════

total_elapsed = (time.time() - overall_start) / 60

print(f"\n\n{'='*65}")
print(f"  COMPLETE — {total_elapsed:.1f} minutes")
print(f"{'='*65}\n")

summary_rows = []
for log in scrape_logs:
    summary_rows.append({
        "Subreddit": f"r/{log['subreddit']}",
        "Posts": f"{log['valid_posts']:,}",
        "Comments": f"{log['total_comments']:,}",
        "Deltas": f"{log['deltabot_posts']:,}",
        "Authors": f"{log['unique_comment_authors']:,}",
        "edited": f"{log['has_edited']:,}",
        "controv.": f"{log['has_controversiality']:,}",
        "Min": log["elapsed_minutes"],
    })

summary_df = pd.DataFrame(summary_rows)
total_c = sum(l["total_comments"] for l in scrape_logs)
total_p = sum(l["valid_posts"] for l in scrape_logs)

print(summary_df.to_string(index=False))
print(f"\nTOTAL: {total_p:,} posts, {total_c:,} comments")

if total_c < 500_000:
    print(f"\n⚠️  {total_c:,} comments — below 3M target.")
elif total_c < 2_000_000:
    print(f"\n📊 {total_c:,} comments — decent, below 3M.")
else:
    print(f"\n✅ {total_c:,} comments — on track for 3M!")

with open(os.path.join(OUT_DIR, "scrape_log_all.json"), "w") as f:
    json.dump(scrape_logs, f, indent=2)

print(f"\n✅ All fields native — no backfill needed")
print(f"Files saved to: {OUT_DIR}")
print(f"Chunk files in: {CHUNKS_DIR}")

Mounted at /content/drive

  r/amitheasshole  — parallel chunked API scrape

  📡 Fetching posts ...
    ⏭️  [2022-2026] all 4 year(s) already on disk
    Total raw posts across chunks: 0

  📡 Fetching comments ...
    ⏭️  [2022-2026] all 4 year(s) already on disk
    Total raw comments across chunks: 57,668

  🔗 Merging and filtering ...
    Merged unique posts: 314,711
    After filtering: 122,617 / 314,711
    Merged unique comments: 20,000
    After filtering: 18,269 / 20,000
    Comments for valid posts: 1

    ✅ r/amitheasshole — 0.5 min
    Posts:    122,617
    Comments: 1
    Top-level: 0 | Replies: 1
    Unique authors:   1
    edited:           1
    controversiality: 1
    has_delta:        0


  COMPLETE — 0.5 minutes

      Subreddit   Posts Comments Deltas Authors edited controv.  Min
r/amitheasshole 122,617        1      0       1      1        1  0.5

TOTAL: 122,617 posts, 1 comments

⚠️  1 comments — below 3M target.

✅ All fields native — no backfill needed
Files save

In [ ]:
import requests
import pandas as pd
import time
import json
import os
import gc
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import glob

# ── Session ─────────────────────────────────────
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20,
                      pool_maxsize=20)
session.mount("https://", adapter)

API_BASE = "https://arctic-shift.photon-reddit.com/api"

# ── Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ════════════════════════════════════════════════
# CONFIG (NEW FOLDER)
# ════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data_resumable_v1"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = ["amitheasshole"]

MIN_COMMENTS = 5
REQUEST_DELAY = 0.1

YEAR_RANGE_CHUNKS = [
    # (2020, 2022),
    (2022, 2024),
    (2024, 2026),
]

POST_FIELDS = [
    "id","author","subreddit","title","selftext","score",
    "created_utc","num_comments","url","over_18",
    "link_flair_text","author_flair_text","edited",
]

COMMENT_FIELDS = [
    "id","author","subreddit","body","score","created_utc",
    "link_id","parent_id","distinguished","author_flair_text",
    "edited","controversiality",
]

print_lock = threading.Lock()
def tprint(msg):
    with print_lock:
        print(msg)

# ════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════

def chunk_path(sub, typ, y0, y1):
    return os.path.join(CHUNKS_DIR, f"{sub}_{typ}_{y0}-{y1}.csv")

def progress_path(path):
    return path.replace(".csv", "_progress.txt")

def is_chunk_complete(path):
    return os.path.exists(path) and os.path.exists(progress_path(path))

# ════════════════════════════════════════════════
# CORE SCRAPER (RESUMABLE)
# ════════════════════════════════════════════════

def paginate_and_save(endpoint, subreddit, fields, y_start, y_end, typ, label):

    chunk = f"{y_start}-{y_end}"
    out = chunk_path(subreddit, typ, y_start, y_end)
    prog = progress_path(out)

    params = {
        "subreddit": subreddit,
        "sort": "asc",
        "limit": 100,
        "after": f"{y_start}-01-01",
        "before": f"{y_end}-01-01",
    }

    total = 0
    results = []
    BATCH = 5000

    # 🔄 RESUME
    if os.path.exists(out):
        try:
            if os.path.exists(prog):
                last_ts = int(open(prog).read().strip())
                params["after"] = last_ts + 1
                tprint(f"      🔄 [{chunk}] resume from {last_ts}")
            else:
                df = pd.read_csv(out, usecols=["created_utc"])
                last_ts = int(df["created_utc"].max())
                params["after"] = last_ts + 1
                tprint(f"      🔄 [{chunk}] resume (scan) {last_ts}")

            total = sum(1 for _ in open(out)) - 1

        except:
            tprint(f"      ⚠️ [{chunk}] resume failed, restarting")

    # 🚀 LOOP
    while True:
        try:
            r = session.get(f"{API_BASE}/{endpoint}", params=params, timeout=60)
            r.raise_for_status()
            data = r.json().get("data", []) or []
        except:
            time.sleep(5)
            continue

        if not data:
            break

        for d in data:
            results.append({f: d.get(f) for f in fields})
            total += 1

        # 💾 SAVE IN BATCH
        if len(results) >= BATCH:
            df = pd.DataFrame(results).drop_duplicates("id")

            df.to_csv(out, mode="a",
                      header=not os.path.exists(out),
                      index=False)

            last_ts = df["created_utc"].max()
            open(prog, "w").write(str(int(last_ts)))

            tprint(f"      💾 [{chunk}] +{len(df):,} (total {total:,})")

            results.clear()
            del df
            gc.collect()

        last_ts = data[-1]["created_utc"]
        params["after"] = last_ts + 1

        if len(data) < 100:
            break

        time.sleep(REQUEST_DELAY)

    # 🧹 FINAL SAVE
    if results:
        df = pd.DataFrame(results).drop_duplicates("id")
        df.to_csv(out, mode="a",
                  header=not os.path.exists(out),
                  index=False)

        last_ts = df["created_utc"].max()
        open(prog, "w").write(str(int(last_ts)))

    tprint(f"      🏁 [{chunk}] DONE ({total:,})")
    return chunk, total

# ════════════════════════════════════════════════
# PARALLEL DRIVER (FIXED)
# ════════════════════════════════════════════════

def fetch_all_chunks(endpoint, sub, fields, typ, label):

    counts = {}

    for rs, re in YEAR_RANGE_CHUNKS:
        years = [(y, y+1) for y in range(rs, re)]

        with ThreadPoolExecutor(max_workers=len(years)) as ex:
            futures = {}

            for y0, y1 in years:
                path = chunk_path(sub, typ, y0, y1)

                if is_chunk_complete(path):
                    n = sum(1 for _ in open(path)) - 1
                    tprint(f"      ⏭️ [{y0}-{y1}] complete ({n:,})")
                    counts[f"{y0}-{y1}"] = n
                    continue

                futures[ex.submit(
                    paginate_and_save,
                    endpoint, sub, fields, y0, y1, typ, label
                )] = f"{y0}-{y1}"

            for f in as_completed(futures):
                k = futures[f]
                try:
                    _, n = f.result()
                    counts[k] = n
                except Exception as e:
                    tprint(f"      ❌ {k}: {e}")
                    counts[k] = 0

    return counts

# ════════════════════════════════════════════════
# MERGE + FILTER (UNCHANGED)
# ════════════════════════════════════════════════

def merge_chunks(sub, typ, fields):
    files = glob.glob(os.path.join(CHUNKS_DIR, f"{sub}_{typ}_*.csv"))
    dfs = [pd.read_csv(f) for f in files if os.path.getsize(f) > 0]
    if not dfs:
        return pd.DataFrame(columns=fields)
    return pd.concat(dfs).drop_duplicates("id")

def filter_posts(df):
    return df[
        (df["selftext"].fillna("") != "") &
        (df["num_comments"] >= MIN_COMMENTS)
    ]

def filter_comments(df):
    return df[
        (df["body"].fillna("") != "")
    ]

# ════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════

for sub in SUBREDDITS:

    print(f"\n=== r/{sub} ===")

    print("\n📡 posts")
    fetch_all_chunks("posts/search", sub, POST_FIELDS, "posts", "posts")

    print("\n📡 comments")
    fetch_all_chunks("comments/search", sub, COMMENT_FIELDS, "comments", "comments")

    print("\n🔗 merging")

    posts = filter_posts(merge_chunks(sub, "posts", POST_FIELDS))
    comments = filter_comments(merge_chunks(sub, "comments", COMMENT_FIELDS))

    print(f"Posts: {len(posts):,}")
    print(f"Comments: {len(comments):,}")

    posts.to_csv(os.path.join(OUT_DIR, f"{sub}_posts.csv"), index=False)
    comments.to_csv(os.path.join(OUT_DIR, f"{sub}_comments.csv"), index=False)

print("\n✅ DONE")
print(f"Saved to: {OUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== r/amitheasshole ===

📡 posts
      💾 [2022-2023] +5,000 (total 5,000)
      💾 [2023-2024] +5,000 (total 5,000)
      💾 [2022-2023] +5,000 (total 10,000)
      💾 [2023-2024] +5,000 (total 10,000)
      💾 [2022-2023] +5,000 (total 15,000)
      💾 [2023-2024] +5,000 (total 15,000)
      💾 [2022-2023] +5,000 (total 20,000)
      💾 [2023-2024] +5,000 (total 20,000)
      💾 [2022-2023] +5,000 (total 25,000)
      💾 [2023-2024] +5,000 (total 25,000)
      💾 [2022-2023] +5,000 (total 30,000)
      💾 [2023-2024] +5,000 (total 30,000)
      💾 [2022-2023] +5,000 (total 35,000)
      💾 [2023-2024] +5,000 (total 35,000)
      💾 [2022-2023] +5,000 (total 40,000)
      💾 [2023-2024] +5,000 (total 40,000)
      💾 [2022-2023] +5,000 (total 45,000)
      💾 [2023-2024] +5,000 (total 45,000)
      💾 [2023-2024] +5,000 (total 50,000)
      💾 [2022-2023] +5,000 (total 50,000)

      💾 [2023-2024] +5,000 (total 4,300,000)
      💾 [2022-2023] +5,000 (total 4,320,000)
      💾 [2023-2024] +5,000 (total 4,305,000)
      💾 [2022-2023] +5,000 (total 4,325,000)
      💾 [2023-2024] +5,000 (total 4,310,000)
      💾 [2022-2023] +5,000 (total 4,330,000)
      💾 [2023-2024] +5,000 (total 4,315,000)
      💾 [2022-2023] +5,000 (total 4,335,000)
      💾 [2023-2024] +5,000 (total 4,320,000)
      💾 [2022-2023] +5,000 (total 4,340,000)
      💾 [2023-2024] +5,000 (total 4,325,000)
      💾 [2022-2023] +5,000 (total 4,345,000)
      💾 [2022-2023] +5,000 (total 4,350,000)
      💾 [2023-2024] +5,000 (total 4,330,000)
      💾 [2023-2024] +5,000 (total 4,335,000)
      💾 [2022-2023] +5,000 (total 4,355,000)
      💾 [2023-2024] +5,000 (total 4,340,000)
      💾 [2022-2023] +5,000 (total 4,360,000)
      💾 [2023-2024] +5,000 (total 4,345,000)
      💾 [2022-2023] +5,000 (total 4,365,000)
      💾 [2023-2024] +5,000 (total 4,350,000)
      💾 [2022-2023] +5,000 (total 4,370,000)
      💾 [2

In [ ]:
import requests
import pandas as pd
import time
import json
import os
import gc
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import glob

# ── Session ─────────────────────────────────────
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries, pool_connections=20,
                      pool_maxsize=20)
session.mount("https://", adapter)

API_BASE = "https://arctic-shift.photon-reddit.com/api"

# ── Drive ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ════════════════════════════════════════════════
# CONFIG (NEW FOLDER)
# ════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data_resumable_v1"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = ["amitheasshole"]

MIN_COMMENTS = 5
REQUEST_DELAY = 0.1

YEAR_RANGE_CHUNKS = [
    # (2020, 2022),
    # (2022, 2024),
    (2024, 2026),
]

POST_FIELDS = [
    "id","author","subreddit","title","selftext","score",
    "created_utc","num_comments","url","over_18",
    "link_flair_text","author_flair_text","edited",
]

COMMENT_FIELDS = [
    "id","author","subreddit","body","score","created_utc",
    "link_id","parent_id","distinguished","author_flair_text",
    "edited","controversiality",
]

print_lock = threading.Lock()
def tprint(msg):
    with print_lock:
        print(msg)

# ════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════

def chunk_path(sub, typ, y0, y1):
    return os.path.join(CHUNKS_DIR, f"{sub}_{typ}_{y0}-{y1}.csv")

def progress_path(path):
    return path.replace(".csv", "_progress.txt")

def is_chunk_complete(path):
    return os.path.exists(path) and os.path.exists(progress_path(path))

# ════════════════════════════════════════════════
# CORE SCRAPER (RESUMABLE)
# ════════════════════════════════════════════════

def paginate_and_save(endpoint, subreddit, fields, y_start, y_end, typ, label):

    chunk = f"{y_start}-{y_end}"
    out = chunk_path(subreddit, typ, y_start, y_end)
    prog = progress_path(out)

    params = {
        "subreddit": subreddit,
        "sort": "asc",
        "limit": 100,
        "after": f"{y_start}-01-01",
        "before": f"{y_end}-01-01",
    }

    total = 0
    results = []
    BATCH = 5000

    # 🔄 RESUME
    if os.path.exists(out):
        try:
            if os.path.exists(prog):
                last_ts = int(open(prog).read().strip())
                params["after"] = last_ts + 1
                tprint(f"      🔄 [{chunk}] resume from {last_ts}")
            else:
                df = pd.read_csv(out, usecols=["created_utc"])
                last_ts = int(df["created_utc"].max())
                params["after"] = last_ts + 1
                tprint(f"      🔄 [{chunk}] resume (scan) {last_ts}")

            total = sum(1 for _ in open(out)) - 1

        except:
            tprint(f"      ⚠️ [{chunk}] resume failed, restarting")

    # 🚀 LOOP
    while True:
        try:
            r = session.get(f"{API_BASE}/{endpoint}", params=params, timeout=60)
            r.raise_for_status()
            data = r.json().get("data", []) or []
        except:
            time.sleep(5)
            continue

        if not data:
            break

        for d in data:
            results.append({f: d.get(f) for f in fields})
            total += 1

        # 💾 SAVE IN BATCH
        if len(results) >= BATCH:
            df = pd.DataFrame(results).drop_duplicates("id")

            df.to_csv(out, mode="a",
                      header=not os.path.exists(out),
                      index=False)

            last_ts = df["created_utc"].max()
            open(prog, "w").write(str(int(last_ts)))

            tprint(f"      💾 [{chunk}] +{len(df):,} (total {total:,})")

            results.clear()
            del df
            gc.collect()

        last_ts = data[-1]["created_utc"]
        params["after"] = last_ts + 1

        if len(data) < 100:
            break

        time.sleep(REQUEST_DELAY)

    # 🧹 FINAL SAVE
    if results:
        df = pd.DataFrame(results).drop_duplicates("id")
        df.to_csv(out, mode="a",
                  header=not os.path.exists(out),
                  index=False)

        last_ts = df["created_utc"].max()
        open(prog, "w").write(str(int(last_ts)))

    tprint(f"      🏁 [{chunk}] DONE ({total:,})")
    return chunk, total

# ════════════════════════════════════════════════
# PARALLEL DRIVER (FIXED)
# ════════════════════════════════════════════════

def fetch_all_chunks(endpoint, sub, fields, typ, label):

    counts = {}

    for rs, re in YEAR_RANGE_CHUNKS:
        years = [(y, y+1) for y in range(rs, re)]

        with ThreadPoolExecutor(max_workers=len(years)) as ex:
            futures = {}

            for y0, y1 in years:
                path = chunk_path(sub, typ, y0, y1)

                if is_chunk_complete(path):
                    n = sum(1 for _ in open(path)) - 1
                    tprint(f"      ⏭️ [{y0}-{y1}] complete ({n:,})")
                    counts[f"{y0}-{y1}"] = n
                    continue

                futures[ex.submit(
                    paginate_and_save,
                    endpoint, sub, fields, y0, y1, typ, label
                )] = f"{y0}-{y1}"

            for f in as_completed(futures):
                k = futures[f]
                try:
                    _, n = f.result()
                    counts[k] = n
                except Exception as e:
                    tprint(f"      ❌ {k}: {e}")
                    counts[k] = 0

    return counts

# ════════════════════════════════════════════════
# MERGE + FILTER (UNCHANGED)
# ════════════════════════════════════════════════

def merge_chunks(sub, typ, fields):
    files = glob.glob(os.path.join(CHUNKS_DIR, f"{sub}_{typ}_*.csv"))
    dfs = [pd.read_csv(f) for f in files if os.path.getsize(f) > 0]
    if not dfs:
        return pd.DataFrame(columns=fields)
    return pd.concat(dfs).drop_duplicates("id")

def filter_posts(df):
    return df[
        (df["selftext"].fillna("") != "") &
        (df["num_comments"] >= MIN_COMMENTS)
    ]

def filter_comments(df):
    return df[
        (df["body"].fillna("") != "")
    ]

# ════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════

for sub in SUBREDDITS:

    print(f"\n=== r/{sub} ===")

    print("\n📡 posts")
    fetch_all_chunks("posts/search", sub, POST_FIELDS, "posts", "posts")

    print("\n📡 comments")
    fetch_all_chunks("comments/search", sub, COMMENT_FIELDS, "comments", "comments")

    print("\n🔗 merging")

    posts = filter_posts(merge_chunks(sub, "posts", POST_FIELDS))
    comments = filter_comments(merge_chunks(sub, "comments", COMMENT_FIELDS))

    print(f"Posts: {len(posts):,}")
    print(f"Comments: {len(comments):,}")

    posts.to_csv(os.path.join(OUT_DIR, f"{sub}_posts.csv"), index=False)
    comments.to_csv(os.path.join(OUT_DIR, f"{sub}_comments.csv"), index=False)

print("\n✅ DONE")
print(f"Saved to: {OUT_DIR}")

Mounted at /content/drive

=== r/amitheasshole ===

📡 posts
      ⏭️ [2024-2025] complete (1,455,710)
      ⏭️ [2025-2026] complete (803,864)

📡 comments
      💾 [2024-2025] +5,000 (total 5,000)
      💾 [2025-2026] +5,000 (total 5,000)
      💾 [2025-2026] +5,000 (total 10,000)
      💾 [2024-2025] +5,000 (total 10,000)
      💾 [2025-2026] +5,000 (total 15,000)
      💾 [2024-2025] +5,000 (total 15,000)
      💾 [2025-2026] +5,000 (total 20,000)
      💾 [2024-2025] +5,000 (total 20,000)
      💾 [2025-2026] +5,000 (total 25,000)
      💾 [2024-2025] +5,000 (total 25,000)
      💾 [2025-2026] +5,000 (total 30,000)
      💾 [2024-2025] +5,000 (total 30,000)
      💾 [2025-2026] +5,000 (total 35,000)
      💾 [2024-2025] +5,000 (total 35,000)
      💾 [2025-2026] +5,000 (total 40,000)
      💾 [2024-2025] +5,000 (total 40,000)
      💾 [2025-2026] +5,000 (total 45,000)
      💾 [2024-2025] +5,000 (total 45,000)
      💾 [2025-2026] +5,000 (total 50,000)
      💾 [2024-2025] +5,000 (total 50,000)
      💾 

      💾 [2025-2026] +5,000 (total 5,195,000)
      💾 [2024-2025] +5,000 (total 5,145,000)
      💾 [2025-2026] +5,000 (total 5,200,000)
      💾 [2024-2025] +5,000 (total 5,150,000)
      💾 [2024-2025] +5,000 (total 5,155,000)
      💾 [2025-2026] +5,000 (total 5,205,000)
      💾 [2024-2025] +5,000 (total 5,160,000)
      💾 [2025-2026] +5,000 (total 5,210,000)
      💾 [2025-2026] +5,000 (total 5,215,000)
      💾 [2024-2025] +5,000 (total 5,165,000)
      💾 [2025-2026] +5,000 (total 5,220,000)
      💾 [2024-2025] +5,000 (total 5,170,000)
      💾 [2025-2026] +5,000 (total 5,225,000)
      💾 [2024-2025] +5,000 (total 5,175,000)
      💾 [2025-2026] +5,000 (total 5,230,000)
      💾 [2024-2025] +5,000 (total 5,180,000)
      💾 [2024-2025] +5,000 (total 5,185,000)
      💾 [2025-2026] +5,000 (total 5,235,000)
      💾 [2025-2026] +5,000 (total 5,240,000)
      💾 [2024-2025] +5,000 (total 5,190,000)
      💾 [2025-2026] +5,000 (total 5,245,000)
      💾 [2024-2025] +5,000 (total 5,195,000)
      💾 [2

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data_resumable_v1"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = [
    "amitheasshole",
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import glob
import os

CHUNKS_DIR = "/content/drive/MyDrive/dissent_project/data_resumable_v1/chunks"

files = sorted(glob.glob(os.path.join(CHUNKS_DIR, "*.csv")))

file_counts = []

for f in files:
    filename = os.path.basename(f)

    try:
        n_rows = sum(1 for _ in open(f)) - 1  # subtract header
    except:
        n_rows = 0

    file_counts.append((filename, n_rows))

# Print nicely
for name, count in file_counts:
    print(f"{name}: {count:,} rows")

amitheasshole_comments_2014-2015.csv: 6,234 rows
amitheasshole_comments_2015-2016.csv: 35,346 rows
amitheasshole_comments_2016-2017.csv: 43,434 rows
amitheasshole_comments_2017-2018.csv: 123,052 rows
amitheasshole_comments_2018-2019.csv: 1,637,710 rows
amitheasshole_comments_2019-2020.csv: 6,571,607 rows
amitheasshole_comments_2020-2021.csv: 6,301,057 rows
amitheasshole_comments_2021-2022.csv: 2,219,644 rows
amitheasshole_comments_2022-2023.csv: 5,213,564 rows
amitheasshole_comments_2023-2024.csv: 4,095,844 rows
amitheasshole_comments_2024-2025.csv: 5,355,427 rows
amitheasshole_comments_2025-2026.csv: 7,814,607 rows
amitheasshole_posts_2014-2015.csv: 2,300 rows
amitheasshole_posts_2015-2016.csv: 8,713 rows
amitheasshole_posts_2016-2017.csv: 8,838 rows
amitheasshole_posts_2017-2018.csv: 17,322 rows
amitheasshole_posts_2018-2019.csv: 148,873 rows
amitheasshole_posts_2019-2020.csv: 1,535,552 rows
amitheasshole_posts_2020-2021.csv: 1,594,562 rows
amitheasshole_posts_2021-2022.csv: 1,082,42

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

OUT_DIR = "/content/drive/MyDrive/dissent_project/data_resumable_v1"
CHUNKS_DIR = os.path.join(OUT_DIR, "chunks")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)

SUBREDDITS = [
    "amitheasshole",
]

Mounted at /content/drive


In [ ]:
def build_final():
    df = merge_chunks("amitheasshole", "comments", COMMENT_FIELDS)
    df = df.drop_duplicates("id")
    df.to_csv("final_comments.csv", index=False)

In [ ]:
import pandas as pd
import glob

def merge_chunks(sub, typ):
    pattern = os.path.join(CHUNKS_DIR, f"{sub}_{typ}_*.csv")
    files = glob.glob(pattern)

    print(f"Found {len(files)} chunk files")

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, low_memory=False)
            if len(df) > 0:
                dfs.append(df)
        except Exception as e:
            print(f"Skipping {f}: {e}")

    if not dfs:
        print("No data found")
        return pd.DataFrame()

    merged = pd.concat(dfs, ignore_index=True)

    print(f"Before dedupe: {len(merged):,}")

    merged = merged.drop_duplicates(subset=["id"])

    print(f"After dedupe: {len(merged):,}")

    return merged

In [ ]:
comments = merge_chunks("amitheasshole", "comments")

comments_path = os.path.join(OUT_DIR, "amitheasshole_comments.csv")
comments.to_csv(comments_path, index=False)

print("Saved to:", comments_path)

Found 12 chunk files
Before dedupe: 14,678,215
After dedupe: 14,678,213
Saved to: /content/drive/MyDrive/dissent_project/data_resumable_v1/amitheasshole_comments.csv


In [ ]:
posts = merge_chunks("amitheasshole", "posts")

posts_path = os.path.join(OUT_DIR, "amitheasshole_posts.csv")
posts.to_csv(posts_path, index=False)

print("Saved to:", posts_path)

Found 12 chunk files
Skipping /content/drive/MyDrive/dissent_project/data_resumable_v1/chunks/amitheasshole_posts_2020-2021.csv: Error tokenizing data. C error: Expected 13 fields in line 275026, saw 19

Before dedupe: 2,253,244
After dedupe: 2,253,244
Saved to: /content/drive/MyDrive/dissent_project/data_resumable_v1/amitheasshole_posts.csv
